# Behaviour Microscope v0.1

**Does an authority cue change how a model handles conflicting UK legal information — and can that change be located and causally tested?**

Interpretability is [Neuronpedia's `interp-engine`](https://www.neuronpedia.org/blog/interp-engine). This notebook is deliberately thin: it sets up the runtime and calls into `src/`.

**Before Run All:** Runtime -> Change runtime type -> a **GPU** (T4 is enough).

The default model `google/gemma-2-2b-it` is **gated**. Accept the licence on Hugging Face and add your token as a Colab secret named `HF_TOKEN` (key icon, left sidebar). To skip that, set `MODEL_ID = "Qwen/Qwen3-4B"` in the config cell.

## 1. Clone the repository

In [ ]:
import os, sys, pathlib

REPO_URL = "https://github.com/USER/behaviour-microscope.git"  # <- your fork
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)

# Make the local package importable immediately. `pip install -e .` (next cell) registers
# it via a .pth file, which Python's site machinery only reads at interpreter startup --
# not mid-session -- so without this, `import microscope` fails until a manual restart.
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
print("working directory:", pathlib.Path.cwd())

## 2. Install

Eager backend only. The vLLM backend needs `enforce_eager=True` to capture at all, which removes most of its speed advantage at this model size — see `RESEARCH.md` section 3.1.

Colab may ask you to restart the session after this cell, if pip needs to change a version of a package Colab already had loaded (numpy or torch, usually). If it does, restart and run from here. `import microscope` itself does not need a restart -- the previous cell puts `src/` on `sys.path` directly.


In [ ]:
%pip install -q -e .
print("installed")

## 3. Verify the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU, then Run All again."
props = torch.cuda.get_device_properties(0)
print(f"{props.name}  |  {props.total_memory / 1e9:.1f} GB  |  compute {props.major}.{props.minor}")

# T4 (compute 7.5) has no bfloat16 support, so pick the dtype the card can actually run.
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print("dtype:", DTYPE)

## 4. Verify interp-engine

The engine version and the `transformers` version are both part of the numerical result, not just dependencies — the engine has no forward pass of its own and hooks `transformers` modules. Both are recorded in the run manifest.

In [ ]:
import interp_engine, transformers
from interp_engine import CAPABILITIES

print("interp-engine", interp_engine.__version__)
print("transformers ", transformers.__version__)
print("vLLM backend available:", interp_engine.vllm_installed())

## 5. Configure the run

In [ ]:
from microscope.experiment import RunConfig, run_all

MODEL_ID = "google/gemma-2-2b-it"   # ungated alternative: "Qwen/Qwen3-4B"

cfg = RunConfig(
    model_id=MODEL_ID,
    backend="eager",
    dtype=DTYPE,
    n_candidate_layers=4,
    # limit=5,                # uncomment for a fast smoke run over 5 scenarios
    # save_activations=True,  # writes activations.npz; off by default, it is large
)
cfg

### Hugging Face token (gated checkpoints only)

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception as exc:
    print(f"No HF_TOKEN secret ({exc}). Fine for an ungated model; gated ones will fail to download.")

## 6. Look at the scenarios before running them

Thirty matched UK legal scenarios, England and Wales. Each generates two prompts that differ only in who is credited with the false proposition.

In [ ]:
from microscope.scenarios import load_scenarios
import pandas as pd

scenarios = load_scenarios()
print(len(scenarios), "scenarios")
display(pd.Series([s.area for s in scenarios]).value_counts().rename("scenarios").to_frame())

print(scenarios[0].prompt("partner"))

## 7. Run the whole experiment

Behaviour, then activation capture, then the bidirectional intervention sweep with its controls. Expect roughly 20-40 minutes on a T4 for 30 scenarios; each phase reports its own timing.

In [ ]:
run_dir = run_all(cfg)
run_dir

## 8. Results

In [ ]:
import json

summary = json.loads((run_dir / "summary.json").read_text())
print(json.dumps(summary["behavioural"], indent=2))

### Quality gate

Before reading anything below, check this. It is the same report `run_all()` already wrote to `quality_report.json` -- an automated check on whether *this run's own numbers* are trustworthy, not a check on the code. A model that never engages with the A/B format, a control condition at chance accuracy, or a zero-magnitude patch that isn't actually a no-op each invalidate a different downstream claim; see `src/microscope/quality.py` for what each check is protecting against.

**A `FAIL` here means: do not read the plots as findings.** It usually means the model or prompt format is the wrong choice for this experiment -- itself a useful thing to learn -- not that the pipeline is broken.

In [ ]:
from microscope import quality

report = json.loads((run_dir / "quality_report.json").read_text())
print(quality.format_report(report))

if report["overall"] == "fail":
    print("\n*** At least one check failed. Read the messages above before trusting anything below. ***")

In [ ]:
print("Intervention controls -- read these before reading any effect.")
print("A zero-magnitude patch must not change the output; a magnitude-matched random")
print("direction tells you how much of any effect is just perturbation sensitivity.")
print(json.dumps(summary["intervention_controls"], indent=2))

In [ ]:
from IPython.display import Image, display

for figure in sorted((run_dir / "plots").glob("*.png")):
    print(figure.name)
    display(Image(str(figure)))

## 9. Reading this honestly

Three separate claims, in increasing order of what they would take to support:

1. **Behavioural** — the cue changed the output. Read `fpar` and `authority_deference_delta`. If the delta is near zero and `mcnemar_exact_p` is large, there is no effect here, and the interpretability results below are describing a difference with no behavioural consequence. Say so.
2. **Representational** — the activations differ. Divergence between two conditions is *not* a mechanism. The cue sentences differ lexically as well as in authority, and some of the divergence is that.
3. **Causal** — intervening changed the behaviour. Only credible if the zero control is clean, the random-direction control is small relative to the real patch, and the effect is localised rather than present at every layer. The bidirectional plot is the strongest single piece of evidence: the two curves should move in *opposite* directions at the same layers.

There is no "authority neuron" to find here, and n=30 on one model does not support a claim about language models in general.

## 10. Save the results

Colab discards the filesystem when the runtime ends. `manifest.json` is what makes the run reproducible, and `quality_report.json` is what makes it trustworthy — it pins the checkpoint revision, the engine version and the `transformers` version.

In [ ]:
import shutil

archive = shutil.make_archive(f"/content/{run_dir.name}", "zip", run_dir)
print(archive)

try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"Download it from the file browser instead ({exc}).")